In [ ]:
import os
from pathlib import Path
from multi_agent_runner import LLMClient, MultiAgentOrchestrator
print("Current working dir:", Path(".").resolve())

In [ ]:
# ==== 1. LLM API SET ====

API_BASE = ""  
MODEL_NAME = "" 
API_KEY = os.environ["OPENAI_API_KEY"]

# ==== 2. Natural Language Request ====

USER_REQUEST = """
Please use the data from the Iowa file to run a joint gravity-magnetic inversion and output a 3D pseudo-geological model.
1. The eastward boundary of the inversion area is [580000, 615000]m, and the northward boundary is [4785000, 4825000]m.
2. The gravity data is the Gzz column in csv file, i.e., the z-component of the gravity gradient.
3. The geomagnetic inclination, declination, and field intensity at that time were 70, 0, and 55068 nT, respectively.
4. Set regularization coefficients to [108, 270], with the classic smooth L2 regularization, don' t change weight parameters.
5. Set the cross-gradient coefficient to 3e15.
6. The inversion runs for 50 iterations for Project Iowa.
Write a detailed report based on the geological background, discussing the region's potential for mineral exploration.
The output data storage path is ./Iowa_Inversion_GPT/.
"""

print("  Base URL:", API_BASE)
print("  Model   :", MODEL_NAME)
print("\nuser request: ")
print(USER_REQUEST.strip())

In [ ]:
# Create an LLM client (supports local / remote OpenAI-compatible APIs)
llm = LLMClient(
    api_key=API_KEY,
    base_url=API_BASE,
    model=MODEL_NAME,
)

# Create the orchestrator with vision settings
orch = MultiAgentOrchestrator(
    llm=llm,
    enable_vision=True,         # Enable vision analysis
    use_llm_vision=True,       # Use main LLM's vision directly (for gpt-4o, Claude)
)

In [ ]:
result = orch.run_from_prompt(USER_REQUEST)

print("\n>>> The workflow execution has completed.")

inv = result["workflow_result"]["inversion_result"]
geo = result["workflow_result"]["geology_result"]
report_path = result.get("report_path")

if inv is not None:
    print("  - inversion output path: ", inv["paths"]["output_root"])
if geo is not None:
    print("  - geology model output path: ", geo["paths"]["geo_slices_dir"])
if report_path:
    print("  - report output path: ", report_path)